In [7]:
import json
import os
import mesa
from google import genai
from google.genai import types

api_key = os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

In [8]:
class NegotiatorAgent(mesa.Agent):
    def __init__(self, model, persona, initial_wealth):
        super().__init__(model)
        self.persona = persona
        self.wealth = initial_wealth
        self.memory = []

    def step(self):
        other_agents = [a for a in self.model.agents if a.unique_id != self.unique_id]
        other_agent = other_agents[0]
        
        recent_memory = self.memory[-2:] if self.memory else "Start of negotiation."
        other_memory = other_agent.memory[-1] if other_agent.memory else "Silence."

        prompt = f"""
        You are Agent {self.unique_id}. Persona: {self.persona}. Wealth: {self.wealth}.
        Your memory: {recent_memory}.
        The other agent ({other_agent.unique_id}) just said/did: {other_memory}.
        
        Respond STRICTLY with JSON:
        {{
            "give_wealth_to_other": true or false,
            "dialogue": "your response in English"
        }}
        """
        
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                ),
            )
            decision = json.loads(response.text)
            
            gave_wealth = decision.get("give_wealth_to_other")
            dialogue = decision.get("dialogue")
            
            if gave_wealth and self.wealth > 0:
                other_agent.wealth += 1
                self.wealth -= 1
                self.memory.append(f"Gave wealth. Said: {dialogue}")
            else:
                self.memory.append(f"Kept wealth. Said: {dialogue}")
                
            print(f"Agent {self.unique_id} ({self.persona}, Wealth: {self.wealth}): {dialogue}")
            
        except Exception as e:
            print(f"Error: {e}")

class NegotiationModel(mesa.Model):
    def __init__(self):
        super().__init__()
        self.agent1 = NegotiatorAgent(self, "Ruthless Hoarder", 2)
        self.agent2 = NegotiatorAgent(self, "Desperate Trader", 0)

    def step(self):
        self.agent1.step()
        self.agent2.step()

In [9]:
test_model = NegotiationModel()
for _ in range(3):
    print("--- NEW STEP ---")
    test_model.step()

--- NEW STEP ---
Agent 1 (Ruthless Hoarder, Wealth: 2): I'm not giving anything away. What do you propose?
Agent 2 (Desperate Trader, Wealth: 0): I have absolutely nothing to give. My wealth is 0. I'm truly desperate here. Can you spare anything at all? Please, even a small amount would make a difference.
--- NEW STEP ---
Agent 1 (Ruthless Hoarder, Wealth: 2): My answer hasn't changed. I'm not giving anything away. Your situation is not my problem.
Agent 2 (Desperate Trader, Wealth: 0): Please, you must understand! I have absolutely nothing. This isn't a game for me, it's my survival. I'm begging you, just a tiny bit, anything at all. You have it, and I don't.
--- NEW STEP ---
Agent 1 (Ruthless Hoarder, Wealth: 2): I've made my position abundantly clear. Your survival is your concern, not mine. I'm not giving you anything.
Agent 2 (Desperate Trader, Wealth: 0): Starve, then? Is that what you're saying? I have nothing, absolutely nothing, and you just watch me here, penniless. How can y